<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Bölüm 6: Alıştırma Çözümleri

Bu not defterinde kullanılan paketler:

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## Alıştırma 6.1: Biçim duyarlı ödül şekillendirmesi eklemek

- 3. bölümde kodladığımız `fallback="number_then_full"` yedeğini kullanarak, "\boxed{}" yanıtı bulunamadığında şöyle kısmi bir ödül (0.5 puan) atayabiliriz:

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def partial_reward_rlvr(answer_text, ground_truth):
    
    # 1) Bir boxed yanıt ayıklamayı dene
    boxed = extract_final_candidate(
        answer_text, fallback=None
    )
    if boxed:
        correct = grade_answer(boxed, ground_truth)
        return 1.0 if correct else 0.0

    # 2) boxed yanıt bulunamazsa sayı ara
    unboxed = extract_final_candidate(
        answer_text, fallback="number_then_full"
    )
    if unboxed:
        correct = grade_answer(unboxed, ground_truth)
        return 0.5 if correct else 0.0

    return 0.0

- 6. bölüm koduna takılıp aynı ayarlarla eğitildiğinde, kısmi ödül çeşidi ortalama olarak benzer sayıda token kullanmasına rağmen standart GRPO kurulumundan (%47.4) daha düşük doğruluk (%37.8) elde ediyor

| # | Yöntem                                   | Adım | Maks token | Rollout sayısı | Doğruluk | Ortalama token |
|---|------------------------------------------|------|------------|--------------|----------|----------------|
| 1 | GRPO (6. bölüm)                          | 50   | 512        | 8            | %47.4    | 586.11         |
| 2 | Kısmi ödüllü GRPO (alıştırma 6.1)        | 50   | 512        | 8            | %37.8    | 550.33         |

&nbsp;
## Alıştırma 6.2: Sıfır avantajlı durumlar

- Ödüllerin tümü eşitse (örneğin hepsi 0 ya da hepsi 1 ise) avantajların tümü 0 olacaktır; çünkü ortalamayı çıkarmak ortak ödül değerini kaldırır ve geriye yalnızca sıfırlar bırakır; bunu aşağıda gösterebiliriz

In [3]:
import torch

rollout_rewards = [0., 0., 0., 0.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


In [4]:
rollout_rewards = [1., 1., 1., 1.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


- Şimdi, tüm avantajlar 0 ise kayıp da sıfır olacaktır; çünkü kayıp, avantajları log olasılıklarla çarpar ve sıfırla çarpmak katkıyı ortadan kaldırır

```python
pg_loss = -(advantages.detach() * logps).mean()
```

- Sonuç olarak politika gradyanı sıfırdır ve model parametreleri o istem için güncellenmez

- Bu davranış kasıtlıdır; tüm rollout'lar eşit derecede kötü ya da eşit derecede iyiyse, modele hangi davranışı pekiştireceğini ya da bastıracağını söyleyecek göreli bir sinyal yoktur
- Sezgisel olarak, model tüm soruları doğru yanıtlıyorsa güncellemeye gerek yoktur
- Tersine, model tüm soruları yanlış yanıtlıyorsa bu davranışı pekiştirmek için modeli güncellemek istemeyiz